# Clase 4 — El ciclo de entrenamiento, y la diferencia entre dibujar y calcar

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Un modelo con suficientes perillas puede pasar
> por todos tus puntos, incluido el ruido. Eso no es aprender: es memorizar tu
> muestra. La única defensa es guardar datos que el modelo nunca vio.

---

**Cuando termines este notebook vas a poder:**

1. Escribir un *training loop* de PyTorch y decir en castellano qué hace cada una
   de sus cinco líneas.
2. Partir tus datos en entrenamiento / validación / test **a mano**, y explicar
   para qué sirve cada montón.
3. Reconocer el sobreajuste (*overfitting*) mirando dos curvas, y frenar a tiempo.
4. Detectar una fuga de datos (*data leakage*) en tu propia partición — y vas a
   ver una, con datos reales del curso, en la sección 6.

**Lo que hace falta traer:** las clases 1, 2 y 3. Nada más.

**Aviso.** Hoy aparece **PyTorch**. No es magia nueva: es exactamente lo que
venís haciendo a mano desde la clase 1, escrito más corto. Lo primero que vamos
a hacer es comprobarlo, no creerlo.

**Cómo se usa este notebook.** Corré las celdas en orden (`Shift + Enter`).
Donde diga `# TODO` te toca a vos. Los `chequear(...)` no explotan: te dicen en
castellano si te salió y, si no, por dónde mirar.

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el
repo y deja todo a mano; en tu máquina no hace prácticamente nada porque ya
tenés los archivos. Corrémela y seguí de largo: no hay nada que entender acá.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys

REPO = "https://github.com/johannperez/dlss"
RAMA = "claude/dl-course-social-scientists-279dd2"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                        REPO, CARPETA], check=True)
    raiz = buscar_raiz(CARPETA)

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab: revisá que la celda haya podido clonar (necesita internet).\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`."
    )

# Dependencias: en Colab casi todas vienen puestas. Instalamos sólo lo que falte.
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],
                       check=True)

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports, semilla y una función de ayuda

Fijamos la semilla en **1983** siempre. No hace al modelo mejor: hace a tu
resultado **reproducible**, que es la condición para poder afirmar algo. Sin
semilla fija, dos corridas del mismo código dan dos resultados distintos y no
sabés cuál reportar. (De esto se ocupa entera la clase 10; desde la 1 lo
venimos haciendo.)

`chequear` es una ayudita de este notebook: verifica algo y te contesta en
castellano, sin tirarte un error rojo de cinco líneas por la cabeza.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

from ssdl import datos, entrenamiento as ent, graficos as g, utils

utils.fijar_semilla(1983)          # numpy + random + torch, todo de una
torch.manual_seed(1983)            # y lo decimos explícito, para que se vea
utils.resumen_del_entorno()


def chequear(condicion, bien, ayuda):
    """Verifica algo y contesta en castellano. Nunca corta el notebook.

    No devuelve nada a propósito: así el notebook no llena la pantalla de `True`.
    """
    print(("✓ " + bien) if condicion else ("✗ Todavía no.\n   " + ayuda))

---

## 1. Tensores en tres minutos

Un **tensor** es una tabla de números. Nada más. El nombre suena imponente pero
la idea es la de siempre:

| forma | cómo se llama | ejemplo en datos sociales |
|---|---|---|
| un número | escalar | la edad de una persona |
| una lista | vector | una serie de tiempo |
| una tabla | matriz | casos × variables |
| una pila de tablas | tensor 3D | documentos × palabras × años |

Lo único que un tensor de PyTorch tiene de más que un array de numpy es que
**se acuerda de las cuentas que le hicieron**. Eso lo vamos a usar en la
sección siguiente y es la razón entera de que exista PyTorch.

In [ ]:
# Un tensor se crea igual que un array de numpy
t = torch.tensor([1.0, 2.0, 3.0])
print("t          =", t)
print("forma      =", t.shape)          # torch dice "shape"; numpy también
print("t * 2      =", t * 2)
print("t.sum()    =", t.sum())

# Una tabla de 3 casos por 2 variables
tabla = torch.tensor([[1.0, 0.0],
                      [0.5, 1.0],
                      [2.0, 3.0]])
print("\nforma de la tabla:", tabla.shape, "→ 3 filas (casos), 2 columnas (variables)")

# Ida y vuelta con numpy: son la misma cosa con otro apellido
a = np.array([1.0, 2.0, 3.0])
print("\nnumpy → torch:", torch.tensor(a, dtype=torch.float32))
print("torch → numpy:", t.numpy())

---

## 2. Autograd: la magia que en realidad ya entendés

La clase pasada calculaste pendientes **con una resta**: movías el parámetro un
poquito para un lado, un poquito para el otro, y mirabas cuánto cambiaba la
loss. Eso se llama derivada numérica y funciona.

PyTorch hace lo mismo pero **exacto y automático**. Le decís a un tensor
"acordate de las cuentas" (`requires_grad=True`), hacés las cuentas, y después
pedís `backward()`. En `.grad` te queda la pendiente.

> **Apostá primero.** Si $y = x^2$ y $x = 3$, ¿cuánto vale la pendiente?
> Escribilo antes de correr la celda.

In [ ]:
# --- la versión de PyTorch --------------------------------------------------
x = torch.tensor(3.0, requires_grad=True)   # "acordate de lo que te haga"
y = x ** 2
y.backward()                                # anotá la pendiente
print(f"PyTorch dice que la pendiente de x² en x=3 es: {x.grad.item()}")

# --- la versión de la clase 3, con una resta --------------------------------
def derivada_numerica(f, punto, h=1e-5):
    """La pendiente medida a lo bruto: cuánto sube dividido cuánto avancé."""
    return (f(punto + h) - f(punto - h)) / (2 * h)

print(f"La resta de la clase 3 dice:                    "
      f"{derivada_numerica(lambda v: v ** 2, 3.0):.6f}")
print("\nMisma cuenta. PyTorch la hace exacta, para millones de perillas a la vez,")
print("y sin que vos escribas una sola derivada. Eso es todo lo que hace autograd.")

### `TODO 1` — probalo con otra función

Cambiá la función a $y = x^3$ y el punto a $x = 2$.

- ¿Cuánto da PyTorch?
- ¿Y la derivada numérica de la clase 3?

*Pista: la celda de abajo ya tiene el esqueleto; sólo hay que cambiar dos cosas.
Si querés anticipar el resultado a mano: la pendiente de $x^3$ es $3x^2$.*

In [ ]:
# TODO 1: cambiá el 2.0 por el punto que quieras y ** 3 por la potencia que quieras.
punto = 2.0
x = torch.tensor(punto, requires_grad=True)
y = x ** 3                       # <-- TODO: probá también ** 2, ** 4, torch.sin(x)
y.backward()

pendiente_torch = x.grad.item()
pendiente_resta = derivada_numerica(lambda v: v ** 3, punto)

print(f"PyTorch: {pendiente_torch:.5f}")
print(f"Resta:   {pendiente_resta:.5f}")

chequear(abs(pendiente_torch - pendiente_resta) < 1e-3,
         "Las dos maneras dan lo mismo. Autograd no es magia: es la cuenta de la clase 3.",
         "Si te dan distinto, fijate que la función de la línea `y = ...` y la del "
         "`lambda` de más abajo sean LA MISMA función. Es el error típico acá.")

---

## 3. El modelo: la red de la clase 2, ahora escrita corta

En la clase 2 armamos a mano una red de una capa oculta: una pila de rampas
(*ReLU*) que se suman. En PyTorch eso se escribe en una línea con
`nn.Sequential`. **Y no vamos a creerle: lo vamos a comprobar.**

La celda de abajo hace las dos cosas —la versión corta y la versión a mano— y
verifica que dan **exactamente** el mismo número. Si alguna vez sentís que
`nn.Sequential` es magia, volvé acá.

In [ ]:
# La red de la clase 2: entra un número, se abre en 16 rampas, se suman, sale un número.
modelo = nn.Sequential(
    nn.Linear(1, 16),   # 16 rampas: cada una con su inclinación y su codo
    nn.ReLU(),          # el doblez (la clase 2 entera)
    nn.Linear(16, 1),   # las alturas con que se suman las 16 rampas
)

print(modelo)
print(f"\nPerillas (parámetros): {utils.contar_parametros(modelo)}")
print("  16 pesos + 16 sesgos de la primera capa, 16 pesos + 1 sesgo de la segunda.")

# --- ¿y si lo escribimos a mano, como en la clase 2? ------------------------
W1, b1 = modelo[0].weight, modelo[0].bias      # (16,1) y (16,)
W2, b2 = modelo[2].weight, modelo[2].bias      # (1,16) y (1,)

entrada = torch.tensor([[0.7], [-1.2], [2.0]])          # tres casos cualesquiera

con_sequential = modelo(entrada)
a_mano = torch.relu(entrada @ W1.T + b1) @ W2.T + b2    # la fórmula de la clase 2

print("\ncon nn.Sequential:", con_sequential.detach().numpy().ravel().round(5))
print("escrito a mano:   ", a_mano.detach().numpy().ravel().round(5))

chequear(torch.allclose(con_sequential, a_mano, atol=1e-6),
         "Son la misma cuenta. nn.Sequential ahorra tipeo, no esconde nada.",
         "Si no coinciden, mirá las transpuestas: PyTorch guarda los pesos como "
         "(salidas, entradas), por eso van los .T")

---

## 4. Los datos de siempre, y el loop

Volvemos a la serie de las clases 1, 2 y 3: **qué porcentaje de las palabras del
discurso presidencial de apertura de sesiones se dedicó al tema *seguridad*, año
por año**, entre 1984 y 2019. Datos reales, 36 casos.

Treinta y seis. Anotá ese número: va a volver a aparecer, y con ganas, en la
segunda mitad.

Como en la clase 3, **centramos el año**. Si no lo hacemos, las dos perillas
quedan tan pegadas que el descenso zigzaguea eternamente. Acá `x` es
*décadas desde 2001*: el año 2011 es `1.0`, el 1991 es `-1.0`.

In [ ]:
anios, seguridad = datos.serie_menciones("seguridad", desde=1984, hasta=2019)

x = torch.tensor((anios - 2001) / 10, dtype=torch.float32).reshape(-1, 1)
y = torch.tensor(seguridad, dtype=torch.float32).reshape(-1, 1)

print(f"{len(anios)} casos, de {int(anios[0])} a {int(anios[-1])}")
print(f"x tiene forma {tuple(x.shape)} → 36 casos, 1 variable cada uno")
print(f"El máximo fue en {int(anios[seguridad.argmax()])}: {seguridad.max():.2f}% de las palabras.")

g.plot_serie(anios, seguridad, etiqueta="menciones a seguridad",
             xlabel="año", ylabel="% de palabras",
             titulo="Los mismos datos desde la clase 1");

### El loop, entero

Estas son **las cinco líneas**. Las cinco. Todo lo que hace un modelo de deep
learning mientras entrena es repetir esto:

1. **PREDECIR** — dibujá con las perillas de ahora.
2. **COMPARAR** — ¿qué tan mal la estás dibujando? (la *loss* de la clase 3)
3. **BORRAR** — tirá lo que anotaste la vuelta pasada.
4. **ANOTAR** — para dónde habría que moverse (*backward*).
5. **MOVER** — recién ahora se mueven las perillas (*step*).

**`backward()` no mueve nada.** Sólo anota. Recién `step()` camina. Si esa
frase te suena rara, abrí `viz/anatomia-del-loop.html` en el navegador y apretá
«una estación» cinco veces: está hecha para eso.

Vocabulario, con un ejemplo numérico y nada más:

- **epoch** (época): una pasada por **todos** tus datos.
- **batch** (lote): el puñado de casos que mirás por vez.
- **iteración**: un paso, o sea una vuelta al loop.

Con 1000 casos y batch de 100 → 10 iteraciones = 1 epoch. Acá tenemos 36 casos y
los miramos todos juntos, así que **1 epoch = 1 iteración** y te podés olvidar
de la distinción por hoy.

In [ ]:
torch.manual_seed(1983)
modelo = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))

criterio = nn.MSELoss()                                    # la loss de la clase 3
optimizador = torch.optim.SGD(modelo.parameters(), lr=0.15)  # el descenso de la clase 3

historia = []
for epoca in range(4000):
    prediccion = modelo(x)                    # 1. PREDECIR  (dibujá)
    perdida = criterio(prediccion, y)         # 2. COMPARAR  (¿qué tan mal?)
    optimizador.zero_grad()                   # 3. BORRAR    (lo de la vuelta pasada)
    perdida.backward()                        # 4. ANOTAR    (para dónde habría que ir)
    optimizador.step()                        # 5. MOVER     (recién acá se mueve)
    historia.append(perdida.item())

    if epoca % 800 == 0:
        print(f"epoch {epoca:5d}   loss {perdida.item():.5f}")

print(f"\nloss final: {historia[-1]:.5f}")

In [ ]:
# ¿Qué dibujó? Al lado, la curva de loss bajando.
with torch.no_grad():
    curva = modelo(x).numpy().ravel()

g.plot_ajuste(anios, seguridad, y_pred=curva,
              xlabel="año", ylabel="% de palabras",
              etiqueta_modelo="lo que dibujó la red (16 rampas)",
              titulo="El reencuentro: el mismo problema, cuatro clases después")

g.plot_serie(np.arange(len(historia)), np.array(historia),
             xlabel="epoch", ylabel="loss (MSE)",
             titulo="Cómo fue bajando el desajuste");

### El reencuentro

Parémonos un segundo acá, porque este gráfico es el pago de cuatro clases.

- En la **clase 1** movías `a` y `b` a ojo con dos sliders.
- En la **clase 2** descubriste que una recta no alcanza y que con rampas sí.
- En la **clase 3** entendiste que se puede bajar sola por el valle.
- Hoy la máquina hizo las tres cosas junta, en veinte líneas.

Los números, para no quedarnos con la emoción:

| modelo | perillas | loss (MSE) |
|---|---|---|
| la **mejor recta posible** (clase 1) | 2 | **0.0146** |
| la red de 16 rampas (hoy) | 49 | **≈ 0.010** |

Un tercio menos de desajuste. Está bien. Pero guardate la desconfianza: parte de
esa mejora es que la red le está copiando el **ruido** a estos 36 puntos. Cuánto,
exactamente, es la segunda mitad de la clase.

In [ ]:
# La mejor recta posible sobre estos mismos datos (mínimos cuadrados, sin misterio).
A = np.vstack([(anios - 2001) / 10, np.ones(len(anios))]).T
coef, *_ = np.linalg.lstsq(A, seguridad, rcond=None)
mse_recta = float(np.mean((A @ coef - seguridad) ** 2))

print(f"La mejor recta:  a={coef[0]:.4f}, b={coef[1]:.4f}   →  MSE {mse_recta:.5f}")
print(f"La red de hoy:                              →  MSE {historia[-1]:.5f}")
print(f"\nEn castellano sustantivo, la recta dice: cada década, las menciones a")
print(f"seguridad suben {coef[0]:.3f} puntos porcentuales. La red no dice nada tan")
print("simple, y esa es exactamente la moneda con que se paga la flexibilidad.")

### `TODO 2` — ¿y si le pongo más neuronas?

Corré la celda de abajo, que entrena la misma red con **4**, **16**, **64** y
**256** rampas, y anotá las cuatro losses.

Antes de correrla, apostá: **¿cuál va a dar la loss más baja?**

*Un detalle honesto que te va a hacer falta el lunes:* cuando cambiás el tamaño
del modelo, el tamaño de paso que servía deja de servir (con 256 rampas y
`lr=0.15` esto explota, probalo). Por eso acá usamos **Adam**, que es descenso
por gradiente con un truco para elegir el paso solo. Por adentro sigue siendo
bajar la colina de la clase 3; es lo que vas a ver en el 90% de los papers.

In [ ]:
def entrenar_simple(x_tr, y_tr, neuronas, epocas=3000, lr=0.01, semilla=1983,
                    x_val=None, y_val=None):
    """El mismo loop de arriba, empaquetado para poder repetirlo. Nada nuevo adentro."""
    torch.manual_seed(semilla)
    m = nn.Sequential(nn.Linear(1, neuronas), nn.ReLU(), nn.Linear(neuronas, 1))
    criterio = nn.MSELoss()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    l_train, l_val = [], []
    for _ in range(epocas):
        perdida = criterio(m(x_tr), y_tr)     # 1 y 2
        opt.zero_grad()                       # 3
        perdida.backward()                    # 4
        opt.step()                            # 5
        l_train.append(perdida.item())
        if x_val is not None:
            with torch.no_grad():
                l_val.append(criterio(m(x_val), y_val).item())
    return m, l_train, l_val


print("neuronas | perillas | loss de entrenamiento")
print("---------|----------|----------------------")
for h in [4, 16, 64, 256]:
    m, l, _ = entrenar_simple(x, y, h)
    print(f"{h:8d} | {utils.contar_parametros(m):8d} | {l[-1]:.5f}")

print("\nMejora. Mejora. Mejora. ¿Entonces le pongo 4000?")
print("Esa pregunta es el resto de la clase.")

### `TODO 3` — rompelo a propósito

Volvé a la celda del loop (la de las cinco líneas) y cambiá `lr=0.15` por
`lr=3.0`. Corré y mirá la curva de loss.

- ¿Reconocés la firma? *Pista: es una de las tres que viste en
  `viz/tasa-de-aprendizaje.html` de la clase 3.*
- ¿Qué le pasó a los números? (`nan` quiere decir "esto ya no es un número":
  la pelota salió despedida tan lejos que la computadora se quedó sin escala.)

La celda de abajo lo hace por vos, para que no tengas que romper la de arriba.

In [ ]:
# La misma red, el mismo problema, un solo número cambiado: el tamaño del paso.
torch.manual_seed(1983)
m_roto = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
opt_roto = torch.optim.SGD(m_roto.parameters(), lr=3.0)   # <-- el desastre
criterio = nn.MSELoss()

primeras = []
for epoca in range(8):
    perdida = criterio(m_roto(x), y)
    opt_roto.zero_grad(); perdida.backward(); opt_roto.step()
    primeras.append(perdida.item())
    print(f"epoch {epoca}: loss {perdida.item():.4g}")

print("\nLa loss SUBE, y después se va a nan. Es la tercera firma de la clase 3:")
print("el paso es tan largo que cada vez la pelota queda más arriba que donde estaba.")
chequear(not np.isfinite(primeras[-1]) or primeras[-1] > primeras[0],
         "Confirmado: con un paso enorme, entrenar empeora el modelo.",
         "Si te bajó la loss, probá subiendo más el lr: cada problema tiene su umbral.")

---

## 5. Datos que el modelo nunca vio

Hasta acá miramos **una sola** cosa: qué tan bien dibuja el modelo sobre los
mismos puntos con los que lo entrenamos. Y ese número **siempre** mejora si le
das más perillas. Siempre. Así que no sirve para decidir nada.

La única defensa es guardar datos que el modelo no vea:

| montón | para qué sirve | cuántas veces se usa |
|---|---|---|
| **entrenamiento** (train) | mover las perillas | todo el tiempo |
| **validación** (val) | decidir **cuántas** perillas y **cuándo parar** | muchas veces |
| **test** | el número que reportás | **una sola vez** |

Lo partimos **a mano**, con índices, para que se vea que no hay magia adentro.
Y —guardá esto— lo vamos a partir **al azar**, que es lo que hace todo el mundo.

En la sección 6 vamos a volver sobre esa decisión y no va a ser lindo.

In [ ]:
# 60 % entrenamiento, 20 % validación, 20 % test. Al azar y con semilla fija.
indices_train, indices_val, indices_test = datos.particion(len(anios), 0.6, 0.2)

x_train, y_train = x[indices_train], y[indices_train]
x_val,   y_val   = x[indices_val],   y[indices_val]
x_test,  y_test  = x[indices_test],  y[indices_test]

print(f"entrenamiento: {len(indices_train)} casos")
print(f"validación:    {len(indices_val)} casos")
print(f"test:          {len(indices_test)} casos   ← este no se toca hasta el final")

print("\nAños que quedaron en el test:      ", sorted(anios[indices_test].astype(int).tolist()))
print("Años que quedaron en validación:  ", sorted(anios[indices_val].astype(int).tolist()))

### Las curvas gemelas: la tijera

Ahora entrenamos midiendo **dos** cosas en cada epoch: la loss sobre lo que el
modelo vio (azul) y la loss sobre lo que no vio (naranja).

> **Apostá primero.** ¿Las dos curvas van a bajar juntas? ¿Una va a bajar y la
> otra no? Escribilo antes de correr.

In [ ]:
def dibujar_tijera(l_train, l_val, titulo):
    """Las dos curvas, con el eje recortado.

    Los primeros epochs son un despeñadero (la loss arranca altísima) y, si los
    dejamos, aplastan todo lo demás contra el piso y la tijera no se ve. Recortar
    el eje para poder mirar lo que importa es legítimo; esconder datos, no. Por
    eso el recorte está acá, a la vista, y no adentro de una función del curso.
    """
    ax = g.curvas_de_entrenamiento(train=l_train, val=l_val, titulo=titulo)
    tope = 1.5 * max(max(l_val[10:]), max(l_train[10:]))
    ax.set_ylim(0, tope)
    return ax


modelo_v, l_train, l_val = entrenar_simple(x_train, y_train, neuronas=16,
                                           epocas=1500, x_val=x_val, y_val=y_val)

mejor_epoca = int(np.argmin(l_val)) + 1        # +1 porque el gráfico cuenta desde 1
print(f"loss de entrenamiento al final: {l_train[-1]:.5f}")
print(f"loss de validación al final:    {l_val[-1]:.5f}")
print(f"\nLa validación tocó su punto más bajo ({min(l_val):.5f}) en la época {mejor_epoca},")
print(f"y de ahí en adelante EMPEORÓ durante {len(l_val) - mejor_epoca} épocas más.")
print("Todo ese rato el modelo siguió 'mejorando'... sobre los datos que ya conocía.")

dibujar_tijera(l_train, l_val, "La tijera: azul lo que vio, naranja lo que no");

**Esta figura se llama «la tijera» y es un diagnóstico.** La vas a ver mil veces.

- Las dos bajan juntas al principio: el modelo está aprendiendo **la forma**.
- La naranja toca fondo y empieza a subir: a partir de ahí el modelo está
  aprendiendo **el ruido de esta muestra**.
- La brecha que se abre entre las dos es, literalmente, cuánto de lo que
  "aprendió" no le sirve para nada más.

**Early stopping** (*parada temprana*) es la consecuencia obvia: guardá el
modelo del punto más bajo de la naranja y quedate con ese. Cinco líneas.

**El detalle incómodo:** si usás la validación para decidir cuándo parar, ese
número ya no es una medida limpia de desempeño — lo usaste para elegir. Por eso
hacen falta **tres** montones y no dos.

In [ ]:
# --- early stopping en cinco líneas -----------------------------------------
torch.manual_seed(1983)
modelo_v = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
criterio, opt = nn.MSELoss(), torch.optim.Adam(modelo_v.parameters(), lr=0.01)

mejor_loss, mejor_estado, mejor_epoca = float("inf"), None, 0
for epoca in range(1500):
    perdida = criterio(modelo_v(x_train), y_train)
    opt.zero_grad(); perdida.backward(); opt.step()
    with torch.no_grad():
        loss_val = criterio(modelo_v(x_val), y_val).item()
    if loss_val < mejor_loss:                                     # ¿mejoró?
        mejor_loss, mejor_epoca = loss_val, epoca + 1             # anotá
        mejor_estado = {k: v.clone() for k, v in modelo_v.state_dict().items()}

modelo_v.load_state_dict(mejor_estado)        # volvé al mejor momento
print(f"Me quedo con el modelo de la época {mejor_epoca}, loss de validación {mejor_loss:.5f}.")
print(f"Las otras {1500 - mejor_epoca} épocas de 'mejora' eran memorización.")

### `TODO 4` — fabricá el sobreajuste a propósito

Ahora al revés: en vez de evitarlo, provocalo. En la celda de abajo:

- subí las neuronas a **256**,
- y bajá el entrenamiento a **15 casos** (`x_train[:15]`).

Corré y describí la tijera que te dio. *Pista: mirá en qué epoch la curva
naranja toca su punto más bajo, y compará la loss de train con la de val al
final.*

In [ ]:
# TODO 4: cambiá estos dos números y volvé a correr.
NEURONAS = 256          # probá 4, 16, 256
CASOS    = 15           # probá 22 (todos), 15, 8

m_over, lt, lv = entrenar_simple(x_train[:CASOS], y_train[:CASOS],
                                 neuronas=NEURONAS, epocas=1500,
                                 x_val=x_val, y_val=y_val)

print(f"{NEURONAS} rampas ({utils.contar_parametros(m_over)} perillas) para {CASOS} casos.")
print(f"loss sobre lo que vio:     {lt[-1]:.5f}")
print(f"loss sobre lo que NO vio:  {lv[-1]:.5f}   ← {lv[-1] / lt[-1]:.0f} veces peor")
print(f"la validación tocó fondo en la época {int(np.argmin(lv)) + 1} y después subió")

dibujar_tijera(lt, lv, f"{NEURONAS} rampas, {CASOS} casos: la tijera de par en par")

chequear(lv[-1] > 3 * lt[-1],
         "Fabricaste un sobreajuste de manual: casi cero de error sobre lo que vio, "
         "y un desastre sobre lo que no.",
         "Si la brecha te quedó chica, subí NEURONAS o bajá CASOS: el sobreajuste "
         "es una relación entre las dos cosas, no una propiedad del modelo solo.")

**Escribí acá tu descripción de la tijera:** _(dos oraciones, en castellano)_

_..._

---

## 6. La escena incómoda: partir al azar

Volvé a la sección 5. Partimos los 36 años **al azar**, como hace todo el mundo,
y nos salió una tijera hermosa y un modelo razonable.

Ahora una pregunta de metodología, de las que hace un revisor:

> **¿Los 36 discursos son 36 casos independientes?**

No. Son **siete presidentes**. Alfonsín dio seis discursos, Menem diez, Cristina
Fernández ocho. Si partís al azar, el modelo estudia cuatro discursos de Menem y
después lo evaluamos con **otro discurso de Menem**. No está prediciendo: está
reconociendo un estilo que ya vio.

Y hay algo peor, y es real: en el corpus fuente, **el discurso de 2005 y el de
2006 son el mismo archivo, palabra por palabra** (mismo `md5sum`; es un error de
la recopilación, está documentado en `datos/README.md`). Corré la celda.

In [ ]:
tabla = datos.serie_completa(1984, 2019)
i2005 = list(tabla["anios"]).index(2005)
i2006 = list(tabla["anios"]).index(2006)

print("           2005        2006")
for k in ["palabras", "seguridad", "inflacion", "educacion", "trabajo", "salud"]:
    print(f"{k:>10}  {tabla[k][i2005]:>9.4f}   {tabla[k][i2006]:>9.4f}")

iguales = all(tabla[k][i2005] == tabla[k][i2006]
              for k in ["palabras", "seguridad", "inflacion", "educacion", "trabajo", "salud"])
print(f"\n¿Son idénticos? {iguales}")
print("No es casualidad ni redondeo: es el mismo discurso cargado dos veces en el corpus.")

In [ ]:
# ¿Dónde cayó cada uno de esos dos gemelos en nuestra partición al azar?
def donde_cayo(anio):
    i = int(anio - 1984)
    if i in indices_train: return "entrenamiento"
    if i in indices_val:   return "validación"
    return "test"

print(f"El discurso de 2005 cayó en: {donde_cayo(2005)}")
print(f"El discurso de 2006 cayó en: {donde_cayo(2006)}")
print("\nO sea: el modelo estudió uno de los dos y lo 'predijo' del otro lado.")
print("Va a acertar ese caso perfecto, y no porque haya aprendido nada.\n")

presidentes = list(tabla["presidente"])
filas = ["|".join(f"{tabla[k][i]:.4f}" for k in
                  ["palabras", "seguridad", "inflacion", "educacion", "trabajo", "salud"])
         for i in range(len(anios))]

print("=== PARTIR AL AZAR ===")
informe = datos.diagnostico_de_fuga(indices_train, indices_test,
                                    grupos=presidentes, textos=filas)

### Las dos maneras honestas de partir estos datos

La pregunta que decide cuál usar no es técnica, es **cuál es tu pregunta de
investigación**:

- *"¿Sirve para el año que viene?"* → **partir por tiempo**. Entrenás con el
  pasado, testeás con el futuro. Cualquier otra cosa es predecir el pasado
  usando el futuro, que es hacer trampa con estilo.
- *"¿Sirve para un presidente que no vi nunca?"* → **partir por grupo**. Un
  presidente entero cae de un solo lado.

Corré y compará los tres diagnósticos.

In [ ]:
print("=== PARTIR POR TIEMPO (train < 2008, val 2008-2013, test 2014-2019) ===")
tr_t, va_t, te_t = datos.particion_temporal(anios, 2008, 2014)
print(f"   {len(tr_t)} / {len(va_t)} / {len(te_t)} casos")
_ = datos.diagnostico_de_fuga(tr_t, te_t, grupos=presidentes, textos=filas)

print("\n=== PARTIR POR GRUPO (cada presidente cae entero de un lado) ===")
tr_g, va_g, te_g = datos.particion_por_grupo(presidentes)
print(f"   {len(tr_g)} / {len(va_g)} / {len(te_g)} casos")
_ = datos.diagnostico_de_fuga(tr_g, te_g, grupos=presidentes, textos=filas)

print("\nLa partición por tiempo mata el duplicado (2005 y 2006 caen los dos en train).")
print("La partición por grupo mata además el problema del presidente compartido,")
print("pero mirá el precio: el test queda con muy pocos casos. No hay almuerzo gratis.")

### Y ahora el número que importa: ¿cuánto te mintió la partición al azar?

Entrenamos **el mismo modelo, con la misma receta**, dos veces: una con la
partición al azar y otra con la partición por tiempo. Y miramos el test.

Agregamos también un **baseline**: predecir siempre el promedio del
entrenamiento. Es el modelo más tonto posible. Si tu red no le gana, tu red no
sirve — y esa comparación falta en muchísimos papers.

> **Apostá primero.** ¿Cuál de las dos particiones va a dar mejor número en test?

In [ ]:
def entrenar_y_testear(idx_tr, idx_va, idx_te, etiqueta, neuronas=16):
    """Entrena con early stopping sobre validación y evalúa UNA vez en test."""
    m, lt, lv = entrenar_simple(x[idx_tr], y[idx_tr], neuronas=neuronas,
                                epocas=1500, x_val=x[idx_va], y_val=y[idx_va])
    # nos quedamos con el mejor momento de validación (early stopping)
    torch.manual_seed(1983)
    m2 = nn.Sequential(nn.Linear(1, neuronas), nn.ReLU(), nn.Linear(neuronas, 1))
    crit, opt = nn.MSELoss(), torch.optim.Adam(m2.parameters(), lr=0.01)
    for _ in range(int(np.argmin(lv)) + 1):     # frenamos donde frenó la validación
        p = crit(m2(x[idx_tr]), y[idx_tr])
        opt.zero_grad(); p.backward(); opt.step()
    with torch.no_grad():
        loss_test = crit(m2(x[idx_te]), y[idx_te]).item()
        promedio = y[idx_tr].mean()
        loss_base = float(((y[idx_te] - promedio) ** 2).mean())
    print(f"{etiqueta}")
    print(f"   mejor loss de validación : {min(lv):.5f}  (época {int(np.argmin(lv)) + 1})")
    print(f"   loss en TEST             : {loss_test:.5f}")
    print(f"   loss del baseline tonto  : {loss_base:.5f}  (predecir siempre el promedio)")
    print(f"   → la red {'LE GANA' if loss_test < loss_base else 'NO le gana'} al baseline\n")
    return loss_test, loss_base


test_azar, base_azar = entrenar_y_testear(indices_train, indices_val, indices_test,
                                          "PARTICIÓN AL AZAR (con fuga)")
test_tiempo, base_tiempo = entrenar_y_testear(tr_t, va_t, te_t,
                                              "PARTICIÓN POR TIEMPO (sin fuga)")

print(f"El test 'al azar' da {test_tiempo / test_azar:.1f} veces mejor que el honesto.")
print("Si publicabas el primero, publicabas un número inflado y no lo sabías.")

> **Ojo con lo que acabamos de hacer.** Miramos el test **dos veces**, a
> propósito, para poder mostrarte el efecto. En tu investigación eso no se hace:
> este test ya está quemado y, si siguiéramos ajustando, necesitaríamos datos
> nuevos. Lo decimos en voz alta porque es exactamente el error que la clase
> quiere que puedas ver.

Y el resultado honesto, el que un revisor querría leer: **con 36 casos, esta red
no le gana a predecir el promedio.** Eso no es un fracaso del notebook: es el
hallazgo. Treinta y seis observaciones son pocas, y decirlo con un número al lado
es mejor trabajo que publicar un 0.014 sin baseline.

---

## 7. El experimento de a dos

De a dos. Elegí **una sola cosa** para cambiar:

- la cantidad de neuronas,
- el learning rate,
- la cantidad de casos de entrenamiento,
- la cantidad de epochs.

**Escribí la hipótesis ACÁ ABAJO, antes de correr nada.** Sí, como en un
preregistro. Por algo será.

---

**Lo que voy a cambiar:** _..._

**Mi hipótesis (qué le va a pasar a la loss de train y a la de validación):**
_..._

**Por qué creo eso:** _..._

---

*(Corré la celda de abajo, volvé y completá:)*

**Lo que pasó:** _..._   **¿Acerté?** _..._

In [ ]:
# Cambiá UNA sola cosa y volvé a correr. Anotá arriba antes de mirar el resultado.
NEURONAS  = 16
EPOCAS    = 1500
LR        = 0.01
CASOS     = len(indices_train)     # bajalo si querés menos datos

m_exp, lt, lv = entrenar_simple(x_train[:CASOS], y_train[:CASOS],
                                neuronas=NEURONAS, epocas=EPOCAS, lr=LR,
                                x_val=x_val, y_val=y_val)

print(f"neuronas={NEURONAS}  epochs={EPOCAS}  lr={LR}  casos={CASOS}")
print(f"   train final : {lt[-1]:.5f}")
print(f"   val final   : {lv[-1]:.5f}")
print(f"   mejor val   : {min(lv):.5f} en la época {int(np.argmin(lv)) + 1}")
print(f"   brecha      : {lv[-1] - lt[-1]:.5f}")

dibujar_tijera(lt, lv, "Tu experimento");

---

## 8. Lo mismo, empaquetado

Todo lo que escribimos a mano está en `ssdl.entrenamiento.entrenar()`: **el
mismo loop de cinco líneas** con contabilidad alrededor. Ahora que sabés qué
hace, podés usarlo sin que sea magia.

`explicar=1` narra las cinco estaciones del primer paso con los números de
verdad. Es la versión en texto de `viz/anatomia-del-loop.html`.

In [ ]:
torch.manual_seed(1983)
modelo_final = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))

historial = ent.entrenar(modelo_final, x_train, y_train,
                         x_val=x_val, y_val=y_val,
                         lr=0.01, epocas=800,
                         optimizador=torch.optim.Adam(modelo_final.parameters(), lr=0.01),
                         paciencia=100,          # early stopping, ya lo escribiste a mano
                         verboso=0, explicar=1)

print(f"\nMejor época de validación: {historial.mejor()}")
print(f"¿Paró temprano? {historial.detenido_temprano}")
print(f"Tardó {utils.formatear_duracion(historial.segundos)}.")

---

## 9. `TODO 5` (de criterio) — cómo se reporta esto

Escribí, en **dos oraciones**, cómo reportarías el resultado de la sección 6 en
un paper. Pensá:

- ¿Qué números incluirías y cuáles no?
- ¿Sobre qué datos es cada número?
- ¿Mencionarías el baseline? ¿Y la partición?

*(Esta pregunta es el ensayo general de la clase 10, que trata entera de esto.)*

**Tu respuesta:**

_..._

---

## 10. Qué te llevás

**Las tres cosas de hoy:**

1. **Entrenar son cinco líneas** que se repiten: predecir, comparar, borrar,
   anotar, mover. `backward()` anota; `step()` mueve. No son lo mismo.
2. **La loss de entrenamiento siempre baja con más capacidad**, así que no sirve
   para decidir nada. La que decide es la loss sobre datos que el modelo no vio.
3. **Partir al azar sólo vale si tus casos son independientes.** Con panel, con
   series de tiempo o con el mismo hablante repetido, partir al azar es hacerse
   trampa sin darse cuenta. Y el test se mira **una sola vez**.

**El vocabulario nuevo** (también está en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *epoch* | una pasada completa por todos tus datos |
| *batch* | el puñado de casos que el modelo mira por vez |
| *training loop* | las cinco líneas que se repiten |
| *tensor* | una tabla de números que se acuerda de las cuentas que le hicieron |
| *autograd* | la maquinaria que calcula todas las pendientes sola |
| *overfitting* | sobreajuste: calcar la muestra en vez de dibujar la forma |
| *train / validation / test* | mover perillas / decidir / reportar |
| *early stopping* | parar cuando la validación deja de mejorar |
| *data leakage* | fuga: el test sabe algo que no debería saber |
| *baseline* | el modelo tonto contra el que hay que ganar |

**Para el lunes, en tu propia investigación.**
Tomá el último dataset con el que trabajaste y contestá tres preguntas por
escrito, antes de tocar un modelo:

1. ¿Mis casos son independientes, o hay una unidad que se repite (persona,
   medio, provincia, sesión)?
2. Si mi pregunta es predictiva, ¿mi partición respeta el tiempo?
3. ¿Cuál es mi baseline tonto, y le gano?

Y cuando leas un paper: **si no dice cómo partió los datos, no reportó nada.**
Preguntalo.

**Lo que queda prometido.** Hoy entraba un número y salía un número. Pero tus
datos son fotos, discursos, secuencias — y la **forma** del dato cambia todo.
La clase que viene: por qué la arquitectura de una red es, en realidad, una
teoría sobre cómo está hecho el mundo.

---

### Para seguir

Las referencias están en `referencias.md`. Empezá por Yarkoni & Westfall (2017):
traduce toda esta clase al vocabulario de las ciencias sociales, sin una sola
línea de código.